# 02 Analysis - Bahn Delay Story

Objective: turn the EDA into defensible findings about trend, train type, station, route, and hour effects.

Run `uv run bahn-pipeline` before this notebook.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

from bahn_delay_story.config import DEFAULT_DATABASE, PROCESSED_DIR

pd.set_option("display.max_columns", 80)
DEFAULT_DATABASE


## Load Processed Tables

In [ ]:
required_outputs = [
    "stops_clean.parquet",
    "station_day_metrics.parquet",
    "train_type_day_metrics.parquet",
    "hourly_delay_metrics.parquet",
    "line_metrics.parquet",
]

missing = [name for name in required_outputs if not (PROCESSED_DIR / name).exists()]
if missing:
    print("Missing processed files:", missing)
    print("Run: uv run bahn-pipeline")

missing


In [ ]:
if not missing:
    with duckdb.connect() as con:
        train_type_day = con.execute(
            "SELECT * FROM read_parquet($path)",
            {"path": str(PROCESSED_DIR / "train_type_day_metrics.parquet")},
        ).df()
        station_day = con.execute(
            "SELECT * FROM read_parquet($path)",
            {"path": str(PROCESSED_DIR / "station_day_metrics.parquet")},
        ).df()
        hourly = con.execute(
            "SELECT * FROM read_parquet($path)",
            {"path": str(PROCESSED_DIR / "hourly_delay_metrics.parquet")},
        ).df()
else:
    train_type_day = pd.DataFrame()
    station_day = pd.DataFrame()
    hourly = pd.DataFrame()

train_type_day.head()


## Trend By Train Type

In [ ]:
if not train_type_day.empty:
    trend = (
        train_type_day
        .groupby(["service_date", "train_type"], as_index=False)
        .agg(
            stop_count=("stop_count", "sum"),
            late_share_6_min=("late_share_6_min", "mean"),
            avg_delay_min=("avg_delay_min", "mean"),
            cancellation_share=("cancellation_share", "mean"),
        )
        .sort_values(["train_type", "service_date"])
    )
else:
    trend = pd.DataFrame()

trend.head()


## Station-Hour Contribution

Use this section to find where late stops concentrate. The final writeup should avoid vague claims like "delays are worse" unless this table identifies who, where, and when.

In [ ]:
if not station_day.empty:
    station_rank = (
        station_day
        .groupby(["station_name", "train_type"], as_index=False)
        .agg(
            stop_count=("stop_count", "sum"),
            late_share_6_min=("late_share_6_min", "mean"),
            avg_delay_min=("avg_delay_min", "mean"),
            p90_delay_min=("p90_delay_min", "mean"),
        )
        .query("stop_count >= 10000")
        .sort_values(["late_share_6_min", "stop_count"], ascending=[False, False])
    )
else:
    station_rank = pd.DataFrame()

station_rank.head(25)


## Candidate Findings

Fill this in only after the tables above are stable.

1. Finding:
2. Evidence:
3. Limitation:
4. Follow-up check: